In [1]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score
)
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
import mlflow

from features.feature_pipeline import BaselineFeatureBuilder, InteractionFeatureBuilder
first_batch_data=np.load("../input/training_batch_with_labels.npz")
second_batch_data=np.load("../input/first_batch_with_labels.npz")

In [6]:
X = np.vstack([first_batch_data['X'], second_batch_data['X']])
y = np.concatenate([first_batch_data['y'], second_batch_data['y']], axis=0)
print("# of interactions:", X.shape[0])
print("# of anomalous and normal users:", np.count_nonzero(y==1), np.count_nonzero(y==0))


df=pd.DataFrame(X)
labels=pd.DataFrame(y)
df.rename(columns={0:"user",1:"item",2:"rating"},inplace=True)
labels.rename(columns={0:"user",1:"label"},inplace=True)
print("# of items:", df['item'].unique().shape[0])

# of interactions: 344839
# of anomalous and normal users: 200 2000
# of items: 996


In [7]:
bf = BaselineFeatureBuilder()
user_features_baseline = bf.transform(df)
user_features_baseline.head()
print(user_features_baseline.shape)


/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:29: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  user_rating_skew=("rating", lambda x: skew(x)),
/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:30: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  user_rating_kurt=("rating", lambda x: kurtosis(x)),


(2200, 17)


In [8]:
# user_features.dtypes
# #  Separate Features and labels
y = labels['label']
X = user_features_baseline.drop(columns=["user"])
print(X.columns, X.shape)
X.isnull().sum()

# Since dataset is imbalanced we startify based on y
# to ensure both train and test set have the same anomaly ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
# Perform standardization of features
scaler = StandardScaler()

# fit -> learn parameters(mean, std) from data
# transform → apply transformation using those parameters
X_train_scaled = scaler.fit_transform(X_train)
# transform only no fitting occurs so that data leakage does not happen, 
# test data uses params of training data and tranforms
X_test_scaled = scaler.transform(X_test)



# Balanced weighting increases penalty for misclassifying anomalies.
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(len(y_pred))
y_prob = model.predict_proba(X_test_scaled)[:,1]
print(y_prob[:5])

Index(['num_unique_items_rated', 'mean_rating', 'median_rating', 'std_rating',
       'min_rating', 'max_rating', 'high_rating_ratio', 'low_rating_ratio',
       'rating_0_pct', 'rating_1_pct', 'rating_4_pct', 'rating_5_pct',
       'user_rating_skew', 'user_rating_kurt', 'rating_entropy',
       'interaction_density'],
      dtype='object') (2200, 16)
440
[0.11154065 0.74378231 0.33637063 0.31634284 0.56458995]


In [9]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Anomaly"],
    columns=["Predicted Normal", "Predicted Anomaly"]
)
print("Confusion Matrix:")
print(cm_df)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)
print("\nROC-AUC:", roc_auc)

# AUPRC (Average Precision Score)
auprc = average_precision_score(y_test, y_prob)
print("AUPRC:", auprc)

Confusion Matrix:
                Predicted Normal  Predicted Anomaly
Actual Normal                316                 84
Actual Anomaly                 9                 31

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.79      0.87       400
           1       0.27      0.78      0.40        40

    accuracy                           0.79       440
   macro avg       0.62      0.78      0.64       440
weighted avg       0.91      0.79      0.83       440


ROC-AUC: 0.8348125000000001
AUPRC: 0.6077607519122183
